# HW10 T5: evolve an image using 50 circles
**Algorithmics 2026 | version 2026-11-02 | helpers, not completed solutions**

A separate notebook for the GA: target, rendering, scoring, your search, live progress and export, in that order.

Run cells from top to bottom. The demonstrations run immediately; assessed
experiment cells are gated by `RUN_T... = False` until your solver is ready.
No downloads, Drive mount or additional helper files are needed. Colab
normally includes NumPy, Matplotlib and Pillow; in local Jupyter install
these helper dependencies if missing.

Your optimiser uses standard Python (`random`, `math`, lists). Do not call
optimisation, evolutionary or regression-fitting packages in T1-T5.
Helper dependencies handle data, scoring and visualisation only. Packages
become allowed in bonus B2. Write your own report; a notebook link is not
a report. Known truth and held-out data must never select a search candidate.


In [ ]:
%matplotlib inline
from pathlib import Path
import random
import math
import numpy as np
import matplotlib.pyplot as plt

OUTPUT = Path('HW10_image_outputs')
OUTPUT.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False})
print("Output folder:", OUTPUT.resolve())
print("NumPy:", np.__version__)


### What does the evaluation budget mean?

**Budget = maximum number of fully scored candidate solutions per run.**
It is not a time limit, generation count, target error or grading points.
A fresh seed/objective/configuration gets a fresh budget.

- One parameter vector scored on ALL training points = **one evaluation**.
- One complete 50-circle image rendered and scored = **one evaluation**.
- Initial candidates and rejected trials count. Recalculating a score
  counts again; reusing a cached score does not. N candidates in a batch
  count as N evaluations, even when computed in one library call.

Example: population 40 and budget 8,000 leave **7,960 evaluations after
initialisation**. With 40 new trials per generation this is 199 further
generations: `40 + 199*40 = 8000`.

Inside your solver, **always obtain search fitness with
`score = run(candidate)`**. This scores and increments `run.evaluations`.
Check `run.remaining` before another evaluation. The helper raises
`BudgetExhausted` if you exceed the limit; raw scoring calls would bypass
it and must not be used for search. The provided runners create the counter
automatically. The saved CSV/JSON logs record actual usage and progress.

**Choose your own evaluation budgets.** Set them before a comparison,
briefly justify them, and keep the same allowance across compared
methods/settings and seeds on the same problem. Different tasks may use
different budgets. More evaluations do not by themselves earn more credit.
The suggested starting values are T1-T3: 8,000; T4: 16,000; T5: 3,000,
not requirements. Use your chosen budget for the main comparisons,
including a partial final generation. T3's target error 0.10 is a milestone,
not an early-stop condition or grading threshold.
Report chosen/used evaluations, your reason, final best fitness and search seconds;
explain any failed or shorter run. Diagnostic landscapes, post-search
held-out checks and final export verification are outside the search
budget and must not select candidates. Plot/display time is separate.


## Image input: choose your file

**Your painting file is not embedded in this notebook. Choose one original
JPG or PNG below.** With no file selected, only a synthetic demo is shown.

**Get an image:** [HW10 image folder](https://drive.google.com/drive/folders/1yJ0MXY9XHislFTIeJKnU0R5dWKHOjZsV).
Download one original JPG/PNG from this course folder, then upload that
file below. A Drive folder URL is not an `IMAGE_PATH` filename.

- **Colab:** set `UPLOAD_IMAGE = True` and run this cell. Click the file
  chooser and select one original painting. Its filename is assigned
  automatically. No Drive mounting is needed.
- **Already uploaded, or local Jupyter:** leave `UPLOAD_IMAGE = False`
  and set `IMAGE_PATH` to the actual filename or path, for example
  `IMAGE_PATH = "Konrad-Mägi-Venice.jpg"` for the supplied Venice painting.
  In Colab, use the Files sidebar to find
  its name; a path on your own computer is not a path inside Colab.
- **Resolution:** set `MAX_SIDE` to your chosen longest side, or `None`
  for original resolution. The source file is never overwritten.

Then run the following cells in order. **Preview the selected image** below
the helper definitions shows the original, working target and renderer demo.


In [ ]:
# Image input: edit these settings, then run this cell.
UPLOAD_IMAGE = False  # True opens the Colab file chooser
IMAGE_PATH = None    # None uses the synthetic demo, not one of the paintings
# After uploading the supplied Venice image, uncomment this line:
# IMAGE_PATH = "Konrad-Mägi-Venice.jpg"
MAX_SIDE = 128       # Suggested longest side; None keeps original resolution

if UPLOAD_IMAGE:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Upload works in Colab; in Jupyter set IMAGE_PATH instead.") from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choose one image, or set IMAGE_PATH to an uploaded filename.")
    IMAGE_PATH = next(iter(uploaded))

if IMAGE_PATH:
    if not Path(IMAGE_PATH).is_file():
        raise FileNotFoundError(f"Image not found: {IMAGE_PATH}. Upload it or correct IMAGE_PATH.")
    print("Selected image:", IMAGE_PATH)
else:
    IMAGE_PATH = None
    print("DEMO ONLY: no painting selected. Set UPLOAD_IMAGE=True or enter IMAGE_PATH above.")
print("Longest side:", MAX_SIDE if MAX_SIDE is not None else "original resolution")


### Experiment budgets: change them here

Choose your own image budget; 3,000 is only a suggested starting value.
Use a smaller debug budget while developing your GA. Briefly justify your
choice and give every compared method and seed the same allowance.
Each run needs 20 evaluations for its initial population, plus enough
trials to observe search progress. Re-run the relevant experiment cells
after changing these settings.


In [ ]:
IMAGE_BUDGET = 3_000  # Suggested T5 budget; choose your own, per 50-circle run
DEBUG_BUDGET = 200    # Five-circle smoke test


## T5. Target and representation

Load your chosen original image with `IMAGE_PATH`, or use the synthetic
landscape for a runnable demonstration. Select the file in **Image input:
choose your file** above. Keep the original file:
the loader reads a separate RGB copy and never overwrites it.

**Choose your own working resolution.** `MAX_SIDE` limits the longest side,
preserving proportions without cropping or stretching. 128 or 256 pixels
is a useful starting point, not a requirement; 64 is suitable for debugging.
`None` uses original resolution. No upscaling is performed. Increasing both
dimensions by two means four times as many pixels to score; elapsed time
also depends on circle count, circle size and implementation.

Use one image and resolution for the main three-seed comparison. Exploring
other sizes is optional. Scores at different resolutions describe different
targets: do not rank algorithms by those raw scores alone. To compare final
genomes from different sizes, render and score each on one shared resolution
after search, reporting this diagnostic separately from training fitness.
Re-run the target and experiment cells when changing the image or resolution.

Candidate: a flat list with seven values per circle in the order
`[x,y,radius,R,G,B,opacity]`. x goes right, y down; positions and colours
are in [0,1], radius in [0.02,0.5] image widths, opacity in [0.05,1].
Draw circles in list order on white. Pixel centres determine circle
membership; there is no antialiasing. Blend in floating point, then round
once to RGB8. Score that final image using mean squared RGB error on [0,1].

The same counter as T1-T4 is embedded here so this notebook stands alone.


In [ ]:
#@title Helper definitions (run this cell)
"""HW10 bookkeeping only: no evolutionary search operators or solvers."""

import csv
from contextlib import contextmanager
import json
import math
from pathlib import Path
import random
from time import perf_counter


class BudgetExhausted(RuntimeError):
    """The next candidate would exceed the agreed evaluation budget."""


def checked_bounds(bounds):
    result = tuple((float(low), float(high)) for low, high in bounds)
    if not result or any(not math.isfinite(low) or not math.isfinite(high)
                         or low >= high for low, high in result):
        raise ValueError("Each bound must be finite and low < high.")
    return result


def checked_candidate(candidate, bounds):
    values = tuple(float(value) for value in candidate)
    if len(values) != len(bounds):
        raise ValueError(f"Expected {len(bounds)} parameters, got {len(values)}.")
    if any(not math.isfinite(value) or not low <= value <= high
           for value, (low, high) in zip(values, bounds)):
        raise ValueError("Candidate is nonfinite or outside the agreed bounds.")
    return values


def initial_population(bounds, size=40, seed=0):
    """Independent initialisation RNG: settings cannot change this population."""
    bounds = checked_bounds(bounds)
    if not isinstance(size, int) or size < 1:
        raise ValueError("Population size must be a positive integer.")
    rng = random.Random(seed)
    return [[rng.uniform(low, high) for low, high in bounds] for _ in range(size)]


class FitnessRun:
    """Count candidate evaluations, keep a copied best, and log search time.

    Call start() immediately before population initialisation, then call this
    object for EVERY evaluated candidate. Call finish() after the search loop.
    A pause() context excludes display/export but refuses fitness evaluations.
    No selection, mutation, crossover or optimisation is implemented here.
    """

    def __init__(self, objective, bounds, budget, *, label, objective_name,
                 seed=0, settings=None, snapshot_every=200):
        if not isinstance(budget, int) or budget < 1:
            raise ValueError("Budget must be a positive integer.")
        if not isinstance(snapshot_every, int) or snapshot_every < 1:
            raise ValueError("Snapshot interval must be a positive integer.")
        self.objective = objective
        self.bounds = checked_bounds(bounds)
        self.budget, self.label = budget, str(label)
        self.objective_name, self.seed = str(objective_name), int(seed)
        self.settings = dict(settings or {})
        self.snapshot_every = snapshot_every
        self.evaluations = 0
        self.best = None
        self.best_fitness = math.inf
        self.history, self.snapshots = [], []
        self._start = None
        self._end = None
        self._paused = 0.0
        self._pause_start = None

    def start(self):
        if self._start is None:
            self._start = perf_counter()
        return self

    @property
    def remaining(self):
        return self.budget - self.evaluations

    @property
    def seconds(self):
        if self._start is None:
            return 0.0
        end = self._end if self._end is not None else perf_counter()
        pending = end - self._pause_start if self._pause_start is not None else 0.0
        return max(0.0, end - self._start - self._paused - pending)

    @contextmanager
    def pause(self):
        if self._pause_start is not None:
            raise RuntimeError("Do not nest pause contexts.")
        active = self._start is not None and self._end is None
        if active:
            self._pause_start = perf_counter()
        try:
            yield
        finally:
            if active:
                self._paused += perf_counter() - self._pause_start
                self._pause_start = None

    def __call__(self, candidate):
        if self.remaining <= 0:
            raise BudgetExhausted(f"Budget of {self.budget} candidates exhausted.")
        if self._end is not None or self._pause_start is not None:
            raise RuntimeError("Cannot score while paused or after finish().")
        values = checked_candidate(candidate, self.bounds)
        self.start()
        score = float(self.objective(values))
        if not math.isfinite(score):
            raise ValueError("Fitness must be finite.")
        self.evaluations += 1
        if score < self.best_fitness:
            self.best_fitness, self.best = score, values
        self.history.append({"evaluation": self.evaluations,
                             "seconds": self.seconds, "fitness": score,
                             "best_fitness": self.best_fitness})
        if self.evaluations == 1 or self.evaluations % self.snapshot_every == 0:
            self.snapshot()
        return score

    def snapshot(self):
        if self.best is not None:
            item = {"evaluation": self.evaluations, "best": list(self.best),
                    "fitness": self.best_fitness}
            if self.snapshots and self.snapshots[-1]["evaluation"] == self.evaluations:
                self.snapshots[-1] = item
            else:
                self.snapshots.append(item)

    def finish(self):
        if self._pause_start is not None:
            raise RuntimeError("Finish outside a pause context.")
        if self._end is None:
            self.start()
            self._end = perf_counter()
            self.snapshot()
        return self.best

    def summary(self, target=None):
        hit = next((row for row in self.history if target is not None
                    and row["best_fitness"] <= target), None)
        return {"label": self.label, "seed": self.seed,
                "objective": self.objective_name, "settings": self.settings,
                "budget": self.budget, "evaluations": self.evaluations,
                "search_seconds": self.seconds,
                "best_fitness": None if self.best is None else self.best_fitness,
                "best": None if self.best is None else list(self.best),
                "target": target, "reached": hit is not None,
                "target_evaluation": None if hit is None else hit["evaluation"],
                "target_seconds": None if hit is None else hit["seconds"]}

    def save(self, stem, target=None):
        """Save every evaluation plus metadata; finish before exporting."""
        if self._end is None:
            raise RuntimeError("Call finish() before saving the run.")
        stem = Path(stem)
        stem.parent.mkdir(parents=True, exist_ok=True)
        with stem.with_suffix(".csv").open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=[
                "evaluation", "seconds", "fitness", "best_fitness"])
            writer.writeheader()
            writer.writerows(self.history)
        data = self.summary(target)
        data.update(bounds=self.bounds, snapshots=self.snapshots)
        stem.with_suffix(".json").write_text(json.dumps(data, indent=2, allow_nan=False) + "\n")


def plot_progress(runs, title="Search progress"):
    import matplotlib.pyplot as plt

    runs = list(runs)
    if not runs or any(not run.history for run in runs):
        raise ValueError("Supply at least one evaluated run.")
    if len({run.objective_name for run in runs}) != 1:
        raise ValueError("Plot different objectives separately: their scores differ.")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), layout="constrained")
    for run in runs:
        for ax, key, label in zip(axes, ("evaluation", "seconds"),
                                  ("Fitness evaluations", "Search time (seconds)")):
            ax.plot([row[key] for row in run.history],
                    [row["best_fitness"] for row in run.history], label=run.label)
            ax.set(xlabel=label, ylabel=f"Best {run.objective_name} (lower is better)")
            ax.grid(alpha=0.2)
    axes[1].legend(fontsize=8)
    fig.suptitle(title)
    return fig


def experiment_table(runs, target=None):
    """All trials, including failures; no winner selection or solver calls."""
    rows = [run.summary(target) for run in runs]
    for row in rows:
        score = row["best_fitness"]
        score_text = "none" if score is None else f"{score:.6f}"
        hit = row["target_evaluation"] if row["reached"] else "not reached"
        target_text = "" if target is None else f"  target={hit}"
        print(f"{row['label']:<24} seed={row['seed']}  fitness={score_text}  "
              f"evals={row['evaluations']}  seconds={row['search_seconds']:.3f}{target_text}")
    return rows


### T5: renderer and scorer (not a GA)


In [ ]:
#@title Helper definitions (run this cell)
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import textwrap
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, PngImagePlugin


RENDERER = "hw10-pixel-centres-alpha-rgb8-v2"


def target_image(size=64):
    """Original synthetic landscape, not a hidden list of solution circles."""
    if not isinstance(size, int) or size < 16:
        raise ValueError("Use an integer image size of at least 16.")
    y, x = np.mgrid[:size, :size] / (size - 1)
    image = np.empty((size, size, 3), dtype=float)
    image[:] = [0.60, 0.82, 0.94]
    image += y[..., None] * np.array([0.18, 0.10, 0.02])
    sun = (x - 0.76)**2 + (y - 0.24)**2 < 0.115**2
    image[sun] = [1.0, 0.76, 0.18]
    distant = y > 0.56 + 0.10 * np.sin(2.5 * np.pi * x + 0.4)
    image[distant] = [0.27, 0.53, 0.43]
    foreground = y > 0.76 + 0.055 * np.cos(3 * np.pi * x)
    image[foreground] = [0.12, 0.35, 0.30]
    river = (y > 0.61) & (np.abs(x - (0.49 + 0.12 * np.sin(7 * y))) < 0.035 + 0.12 * (y - 0.61))
    image[river] = [0.30, 0.66, 0.84]
    return np.rint(np.clip(image, 0, 1) * 255).astype(np.uint8)


def circle_bounds(circles=50):
    if not isinstance(circles, int) or circles < 1:
        raise ValueError("Use a positive integer circle count.")
    return [(0, 1), (0, 1), (0.02, 0.5), (0, 1), (0, 1), (0, 1), (0.05, 1)] * circles


def load_rgb(path):
    """Load a separate RGB copy, respecting EXIF orientation; never rewrite the source."""
    with Image.open(path) as source:
        oriented = ImageOps.exif_transpose(source).convert("RGBA")
        white = Image.new("RGBA", oriented.size, (255, 255, 255, 255))
        return np.array(Image.alpha_composite(white, oriented).convert("RGB"))


def prepare_target(original, max_side=128):
    """Return a working copy, preserving aspect ratio; None keeps full resolution."""
    original = checked_target(original)
    if max_side is None:
        return original.copy()
    if type(max_side) is not int or max_side < 1:
        raise ValueError("Use a positive integer longest side, or None for original resolution.")
    height, width = original.shape[:2]
    if max(height, width) <= max_side:
        return original.copy()
    scale = max_side / max(height, width)
    dimensions = (max(1, round(width * scale)), max(1, round(height * scale)))
    return np.array(Image.fromarray(original).resize(dimensions, Image.Resampling.LANCZOS))


def render_circles(genome, size=64):
    """White RGB canvas; circles at pixel centres, alpha blended in list order.

    size is an integer for a square, or (height, width). x goes right and y
    down, each relative to its own axis. Radius is in image-width units. No edge
    antialiasing. Blend in floating point and quantise once to RGB8 at the end.
    """
    dimensions = (size, size) if type(size) is int else size
    if (not isinstance(dimensions, (tuple, list)) or len(dimensions) != 2
            or any(type(value) is not int or value < 1 for value in dimensions)):
        raise ValueError("Use a positive integer size or (height, width).")
    height, width = dimensions
    genome = tuple(genome)
    if not genome or len(genome) % 7:
        raise ValueError("A genome has seven values per circle.")
    genome = checked_candidate(genome, circle_bounds(len(genome) // 7))
    image = np.ones((height, width, 3), dtype=float)
    x_centres = np.arange(width) + 0.5
    y_centres = np.arange(height) + 0.5
    for i in range(0, len(genome), 7):
        x, y, radius, red, green, blue, alpha = genome[i:i + 7]
        cx, cy, radius_px = x * width, y * height, radius * width
        left = max(0, int(np.floor(cx - radius_px)))
        right = min(width, int(np.ceil(cx + radius_px)))
        top = max(0, int(np.floor(cy - radius_px)))
        bottom = min(height, int(np.ceil(cy + radius_px)))
        mask = ((y_centres[top:bottom, None] - cy)**2
                + (x_centres[None, left:right] - cx)**2 <= radius_px**2)
        region = image[top:bottom, left:right]
        region[mask] = (1 - alpha) * region[mask] + alpha * np.array([red, green, blue])
    return np.rint(np.clip(image, 0, 1) * 255).astype(np.uint8)


def checked_target(target):
    target = np.asarray(target)
    if (target.dtype != np.uint8 or target.ndim != 3 or target.shape[2] != 3
            or target.shape[0] < 1 or target.shape[1] < 1):
        raise ValueError("Target must be a nonempty uint8 RGB image.")
    return target


def image_score(genome, target):
    target = checked_target(target)
    image = render_circles(genome, size=target.shape[:2])
    residual = (image.astype(float) - target.astype(float)) / 255.0
    return float(np.mean(residual**2))


def image_digest(target):
    target = checked_target(target)
    return hashlib.sha256(str(target.shape).encode("ascii") + target.tobytes()).hexdigest()


def run_image(solver, target, *, seed=0, circles=50, population=20, budget=3000, progress=None):
    """Call YOUR GA. It must score the initial population and all trials via run()."""
    if not isinstance(population, int) or population < 2:
        raise ValueError("Use a population of at least two candidates.")
    target = checked_target(target).copy()
    bounds = circle_bounds(circles)
    run = FitnessRun(lambda genome: image_score(genome, target), bounds, budget,
                     label=f"GA {circles} circles, {target.shape[1]}x{target.shape[0]}, seed {seed}",
                     objective_name="RGB MSE", seed=seed,
                     settings={"circles": circles, "population": population,
                               "target_sha256": image_digest(target), "renderer": RENDERER,
                               "height": target.shape[0], "width": target.shape[1]},
                     snapshot_every=200)
    if run.budget < population:
        raise ValueError("Budget must cover the entire initial population.")
    run.start()
    initial = initial_population(bounds, population, seed)
    solver(run, initial, seed=10000 + seed, progress=progress)
    run.finish()
    if run.evaluations == 0:
        raise RuntimeError("Your GA did not evaluate any candidates.")
    if progress is not None:
        progress(run, force=True)
    return run


### T5: static and optional live views


In [ ]:
def show_images(images, title="", columns=3):
    items = list(images.items())
    if not items:
        raise ValueError("Supply at least one labelled image.")
    columns = min(columns, len(items))
    rows = (len(items) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 3.8 * rows),
                             squeeze=False, layout="constrained")
    for ax, (label, image) in zip(axes.flat, items):
        ax.imshow(image, interpolation="nearest")
        ax.set_title(label, fontsize=10, wrap=True)
        ax.axis("off")
    for ax in list(axes.flat)[len(items):]:
        ax.axis("off")
    fig.suptitle(title)
    return fig


def plot_image_evolution(run, target):
    if run.best is None:
        raise ValueError("Run has no evaluated candidates.")
    if run.settings.get("target_sha256") != image_digest(target):
        raise ValueError("Use the same target and resolution as this run.")
    initial_count = run.settings.get("population", 1)
    snapshots = [snap for snap in run.snapshots if snap["evaluation"] >= initial_count] or run.snapshots
    if len(snapshots) > 5:
        snapshots = [snapshots[i] for i in np.linspace(0, len(snapshots) - 1, 5, dtype=int)]
    images = {"Target": target}
    for snap in snapshots:
        images[f"Eval {snap['evaluation']}: MSE {snap['fitness']:.5f}"] = render_circles(snap["best"], target.shape[:2])
    return show_images(images, f"T5: {run.settings['circles']} circles, best-so-far images", columns=3)


class LiveImageView:
    """Optional notebook display. Call after scoring, never inside the objective."""

    def __init__(self, target, every=200):
        if not isinstance(every, int) or every < 1:
            raise ValueError("Display interval must be a positive integer.")
        self.target, self.every = checked_target(target).copy(), every
        self._last, self._handle = -1, None

    def __call__(self, run, force=False):
        if run.best is None or run.evaluations == self._last:
            return
        if not force and run.evaluations % self.every:
            return
        if run.settings.get("target_sha256") != image_digest(self.target):
            raise ValueError("Use the same target and resolution as this run.")
        from IPython.display import display

        with run.pause():
            fig, axes = plt.subplots(1, 4, figsize=(13, 3.5), layout="constrained")
            for ax, image, label in zip(axes[:2], [self.target, render_circles(run.best, self.target.shape[:2])],
                                        ["Target", f"Best at {run.evaluations}: {run.best_fitness:.5f}"]):
                ax.imshow(image, interpolation="nearest")
                ax.set_title(label, fontsize=10)
                ax.axis("off")
            for ax, key, label in zip(axes[2:], ("evaluation", "seconds"), ("Evaluations", "Search seconds")):
                ax.plot([row[key] for row in run.history], [row["best_fitness"] for row in run.history])
                ax.set(xlabel=label, ylabel="Best RGB MSE")
                ax.grid(alpha=0.2)
            if self._handle is None:
                self._handle = display(fig, display_id=True)
            else:
                self._handle.update(fig)
            plt.close(fig)
            self._last = run.evaluations


### Preview the selected image


In [ ]:
original = load_rgb(IMAGE_PATH) if IMAGE_PATH is not None else target_image(size=256)
target = prepare_target(original, max_side=MAX_SIDE)
target_id = image_digest(target)
height, width = target.shape[:2]
Image.fromarray(target).save(OUTPUT / f"T5_target_{width}x{height}_{target_id[:12]}.png")
print("Source:", IMAGE_PATH if IMAGE_PATH is not None else "Synthetic demonstration")
print("Original (width, height):", (original.shape[1], original.shape[0]))
print("Working (width, height):", (width, height), "Target SHA256:", target_id)
# This random candidate is a renderer demo, NOT an evolved solution.
example_genome = initial_population(circle_bounds(50), 1, seed=0)[0]
example_image = render_circles(example_genome, size=target.shape[:2])
scoring_demo = FitnessRun(lambda genome: image_score(genome, target), circle_bounds(50),
                         3, label="Counter demonstration only", objective_name="RGB MSE")
example_score = scoring_demo(example_genome)
scoring_demo.finish()
print(f"One complete image scored: used {scoring_demo.evaluations}/{scoring_demo.budget}; "
      f"remaining {scoring_demo.remaining}. Demo only, not the final experiment.")
fig = show_images({"Original": original, f"Working target: {width}x{height}": target,
                   "Random 50-circle illustration": example_image},
                  f"Renderer demonstration only; MSE {example_score:.5f}")
fig.savefig(OUTPUT / "T5_renderer_demo.png", dpi=140)
plt.show()


## T5. Your genetic algorithm

Fill `student_ga`. Score the initial population and EVERY evaluated
offspring using `run(genome)`. Cache scores for unchanged individuals.
Write your own parent selection, whole-circle crossover, mutation and
elitism using standard Python. A circle is `genome[7*i:7*i+7]`.
Retain the best candidate, clip mutated values to `run.bounds`, and stop
at the budget (`run.remaining`). Use `random.Random(seed)` for variation.

After initial scoring, call `run.snapshot()` so the best initial image
is retained. After scoring an offspring, optionally call
`progress(run)` when a progress object is supplied. It handles the
checkpoint interval and excludes display overhead from search time.
This is a display hook, not an evolutionary operator.


In [ ]:
def student_ga(run, population, *, seed=0, progress=None):
    # Implement selection, crossover, mutation and elitism here.
    raise NotImplementedError("Write your GA before enabling experiments.")

RUN_DEBUG = False
if RUN_DEBUG:
    debug_run = run_image(student_ga, target, circles=5, population=20,
                          budget=DEBUG_BUDGET, progress=LiveImageView(target, every=50))
    plot_progress([debug_run], "Five-circle debugging run, not the final experiment")
    plt.show()


## T5. Required experiment and fitness tracking

Run 50 circles, population 20, your chosen `IMAGE_BUDGET`, seeds 0/1/2,
using the same chosen image and resolution. Do not
initialise with a traced target or previously fitted genome. You need
not reproduce the target perfectly. Compare visible progress with the
numeric score; image approximation is the task, not recovering an exact
hidden original. The live view is optional: leave `LIVE = False` for
simpler batch execution. Static plots are always available afterwards.


In [ ]:
RUN_T5 = False
LIVE = False
image_runs = []
if RUN_T5:
    for seed in (0, 1, 2):
        view = LiveImageView(target) if LIVE else None
        run = run_image(student_ga, target, seed=seed, circles=50,
                        budget=IMAGE_BUDGET, progress=view)
        image_runs.append(run)
        plot_image_evolution(run, target)
        plt.show()
    plot_progress(image_runs, "T5: all three seeds")
    plt.show()
    image_rows = experiment_table(image_runs)
else:
    print("T5 waits for your student_ga implementation.")


### Optional: more circles, if you have time

Keep your 50-circle results, then try 100 or 200 circles. Start with the
same target, resolution, GA settings, evaluation budget and seed(s).
Compare visual detail, fitness and elapsed time. Each candidate still
counts as one evaluation, but takes more rendering work; the search has
`7 * circles` parameters. More circles need not improve the result within
the same budget. Label any larger-budget runs separately.

This optional cell illustrates one seed, not reliability across seeds.
It is not another required task or a separate bonus. The exporter below
includes circle count in filenames, keeping the baseline and extra run.


In [ ]:
RUN_MORE_CIRCLES = False
EXTRA_CIRCLES = 100  # Optional: try 100 or 200 alongside the 50-circle baseline
extra_image_runs = []
if RUN_MORE_CIRCLES:
    extra_run = run_image(student_ga, target, seed=0, circles=EXTRA_CIRCLES,
                          budget=IMAGE_BUDGET)
    extra_image_runs.append(extra_run)
    plot_image_evolution(extra_run, target)
    plt.show()
    baseline = [run for run in image_runs
                if run.seed == 0 and run.budget == IMAGE_BUDGET
                and run.settings["target_sha256"] == image_digest(target)]
    experiment_table(baseline + extra_image_runs)
    plot_progress(baseline + extra_image_runs, "Circle count: seed 0, equal evaluation budgets")
    plt.show()


## T5. Export and verify

Export every run, not just the most successful seed. A raw PNG at your
working resolution contains exactly the scored pixels; a separate preview adds your name,
recomputed score and UTC timestamp. JSON holds the genome and settings;
CSV holds every fitness evaluation. The verifier regenerates pixels and
score independently. Include your GA operator settings in `run.settings`
before exporting. Save the paired `_target.png` too: it contains the exact
working target used for verification. Filenames identify target, resolution,
circle count, seed and budget. Export before changing the target. The target
itself is never counted as a candidate. A full-resolution render of the
same genome can be displayed with `render_circles(run.best, original.shape[:2])`;
it is not the scored result unless training used that resolution.


In [ ]:
def export_image_run(run, target, name, stem="HW10_image_outputs/best"):
    if not str(name).strip() or len(str(name).strip()) > 100:
        raise ValueError("Use a nonempty participant name up to 100 characters.")
    if run.best is None or run._end is None:
        raise ValueError("Finish a nonempty run before exporting.")
    if run.settings.get("target_sha256") != image_digest(target):
        raise ValueError("Target differs from the image used in this run.")
    score = image_score(run.best, target)
    if abs(score - run.best_fitness) > 1e-12:
        raise ValueError("Recomputed score differs from the logged score.")
    stem = Path(stem)
    stem.parent.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now(timezone.utc).isoformat(timespec="seconds")
    height, width = target.shape[:2]
    caption = (f"{name.strip()} | RGB MSE: {score:.8f} | {width}x{height}, "
               f"{len(run.best) // 7} circles\n{timestamp} | evaluations: {run.evaluations}")
    image = render_circles(run.best, (height, width))
    png_info = PngImagePlugin.PngInfo()
    png_info.add_text("Description", caption)
    Image.fromarray(image).save(stem.with_suffix(".png"), pnginfo=png_info)
    Image.fromarray(target).save(stem.parent / (stem.name + "_target.png"))
    result = {"schema": "hw10-circles-v2", "renderer": RENDERER,
              "name": name.strip(), "timestamp_utc": timestamp,
              "target_sha256": image_digest(target), "height": height, "width": width,
              "circles": len(run.best) // 7, "genome": list(run.best), "score": score,
              "seed": run.seed, "evaluations": run.evaluations, "budget": run.budget,
              "settings": run.settings}
    stem.with_suffix(".json").write_text(json.dumps(result, indent=2, allow_nan=False) + "\n")
    run.save(stem.parent / (stem.name + "_log"))
    fig = show_images({"Target": target, "Best approximation": image},
                      "\n".join(textwrap.fill(line, 95) for line in caption.splitlines()), columns=2)
    fig.savefig(stem.parent / (stem.name + "_preview.png"), dpi=140)
    plt.close(fig)
    return result


def verify_image_export(stem, target):
    stem = Path(stem)
    record = json.loads(stem.with_suffix(".json").read_text())
    if record["schema"] != "hw10-circles-v2" or record["renderer"] != RENDERER:
        raise ValueError("Unknown genome schema or renderer.")
    if (record["target_sha256"] != image_digest(target)
            or (record["height"], record["width"]) != target.shape[:2]):
        raise ValueError("Wrong target image.")
    with Image.open(stem.parent / (stem.name + "_target.png")) as saved_target:
        if not np.array_equal(np.asarray(saved_target.convert("RGB")), target):
            raise ValueError("Saved target differs from the image used in this run.")
    if len(record["genome"]) != 7 * record["circles"]:
        raise ValueError("Wrong circle count.")
    image = render_circles(record["genome"], (record["height"], record["width"]))
    score = image_score(record["genome"], target)
    if not np.isfinite(record["score"]) or abs(score - record["score"]) > 1e-12:
        raise ValueError("Incorrect claimed score.")
    if not np.array_equal(np.asarray(Image.open(stem.with_suffix(".png")).convert("RGB")), image):
        raise ValueError("PNG does not match the saved genome.")
    return score


In [ ]:
PARTICIPANT = "Your name"
if image_runs or extra_image_runs:
    if PARTICIPANT == "Your name":
        print("Set PARTICIPANT to your name before exporting.")
    else:
        for run in image_runs + extra_image_runs:
            settings = run.settings
            stem = OUTPUT / (f"T5_{settings['width']}x{settings['height']}_"
                f"{settings['target_sha256'][:12]}_circles_{settings['circles']}_"
                f"seed_{run.seed}_budget_{run.budget}")
            export_image_run(run, target, PARTICIPANT, stem)
            print(stem, "verified MSE:", verify_image_export(stem, target))


## Optional comparison and report

B2 may compare a packaged method on this same representation, renderer,
target and budget. Route all candidates through the counter, report three
seeds, and distinguish faster implementation from better search.

Report the image source, original and working dimensions, circle count,
target, initial/intermediate/final images, fitness histories and all final
scores. Submit JSON, PNG, working target PNG and logs with your code; put selected
figures and your interpretation in a self-contained PDF/HTML report.
